#### Step 1: Import Spark functions
This cell imports Spark SQL functions that are reused across ingestion notebooks.

What this step is intended to do:
* Make Spark helper functions available in this setup notebook
* Support reusable transformations such as file metadata and timestamps
* Centralize common imports so downstream notebooks can call this notebook with `%run`


#### Step 2: Define shared environment variables
This cell stores the catalog, schema, storage, and layer names used across the project.

What this step is intended to do:
* Keep commonly used paths and namespaces in one place
* Reduce hard-coded values in ingestion notebooks
* Make it easier to switch environments later if needed


#### Step 3: Create a reusable metadata helper
This cell defines a helper function that appends standard metadata columns to a DataFrame.

What this step is intended to do:
* Capture the source file path for lineage
* Add an ingestion timestamp for auditability
* Standardize metadata enrichment across raw ingestion notebooks


In [ ]:

# Import Spark SQL functions so they can be reused in metadata transformations.
# This module supports adding file paths, timestamps, and other column expressions.
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import TimestampNTZType
import os
import sys


In [ ]:
project_root = os.path.abspath("../..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
# Define project-wide environment variables for Unity Catalog namespaces and storage layers.
# These values are reused by all ingestion and transformation notebooks via %run.
volume = 'Volumes'        # Unity Catalog Volumes root folder
schema = 'landing'        # Landing zone schema for raw data
file = 'files'            # Files folder within the volume
bronze = 'bronze'         # Bronze layer schema name
silver = 'silver'         # Silver layer schema name
gold = 'gold'             # Gold layer schema name
ext_location = 'abfss://company-dev@stcompanydev.dfs.core.windows.net/'

In [ ]:
# Define a reusable helper function that adds standard metadata columns to any DataFrame.
# 'path' captures the source file name for lineage, and 'timestamp' records when the data was ingested.
def add_columns(df):
    return df.withColumns({'Path': F.col('_metadata.file_path'), 'Ingestion_Timestamp': F.current_timestamp(), 'batch_id': F.lit(f"{batch_id}")})